In [10]:
import torch

x = torch.tensor(3.0, requires_grad=True) # without this requres_grad we can't use y.backward()

y = x ** 2

print(x)
print(y)

# Till here function behave like normal maths and from now on we are actually calculating the gradient which
# we can get by x.grad later for calculations
y.backward()
print(x.grad)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(6.)


### 🔄 How Computational Graphs Work

#### 1. Forward Pass (Function Evaluation)
Our computation flows from independent variables to the final output:
$$
x \xrightarrow{\text{square}} x^2 \xrightarrow{\text{assign}} y
$$

For a concrete example where $x = 3$:
* $y = x^2 = 3^2 = 9$

#### 2. Backward Pass (Gradient Calculation)
We traverse the computational graph in reverse to find out how much a tiny change in our inputs affects the output:
$$
y \rightarrow x
$$

Using calculus, the derivative (gradient) of $y$ with respect to $x$ is:
$$
\frac{dy}{dx} = \frac{d}{dx}(x^2) = 2x
$$

Evaluating this at $x = 3$:
$$
\frac{dy}{dx} \bigg|_{x=3} = 2(3) = 6
$$

This fundamental mechanism is called **Backpropagation**, and it allows PyTorch to compute complex derivatives automatically.

In [11]:
# Let's make things interesting with more variables
x = torch.tensor(2.0, requires_grad=True)

y = 3*x**2 + 2*x + 1
print(y)  # should be 17

y.backward()
print(x.grad)  # should 14

tensor(17., grad_fn=<AddBackward0>)
tensor(14.)


### 🧠 Why is Autograd Crucial for Machine Learning?

This is where theory connects directly to training neural networks.

Consider a single **Perceptron** unit:
$$
z = w_1x_1 + w_2x_2 + b
$$

During training, our information flows in a structured pipeline:
$$
\text{Input} \longrightarrow \text{Weights/Biases} \longrightarrow \text{Prediction} \longrightarrow \text{Loss Evaluation}
$$

To improve the model's accuracy, we need to know exactly how to adjust our parameters ($w_1$, $w_2$, and $b$) to minimize the **Loss**.

Mathematically, this means calculating the partial derivatives of the Loss with respect to each parameter:

* **Weight 1 Gradient:**
$$
\frac{\partial \text{Loss}}{\partial w_1}
$$
* **Weight 2 Gradient:**
$$
\frac{\partial \text{Loss}}{\partial w_2}
$$
* **Bias Gradient:**
$$
\frac{\partial \text{Loss}}{\partial b}
$$

Instead of calculating these complex derivatives by hand for millions of parameters, PyTorch's **Autograd** system automates this entire process dynamically.

### 🔁 The Deep Learning Training Loop Lifecycle

Below is the sequential cycle that every modern neural network executes during training:

```
          [ FORWARD PASS ]                     [ BACKWARD PASS ]
          
             Input Data                            Loss Value
                 │                                     │
                 ▼                                     ▼
           Hidden Layers                           Gradients
                 │                                     │
                 ▼                                     ▼
            Predictions                          Layer Weights
                 │                                     │
                 ▼                                     ▼
             Loss Value                           Weight Update
```

---

### ⏱️ Step-by-Step Training Phase

1. **Forward Propagation**
   * Feed inputs through the model to obtain predictions.
2. **Loss Computation**
   * Calculate a scalar metric indicating how far off predictions are from the ground truth.
3. **Backpropagation**
   * Trace backwards through the computational graph to evaluate parameter gradients.
4. **Optimization (Gradient Descent)**
   * Update weights and biases in the direction that minimizes loss:
   $$
   W_{\text{new}} = W_{\text{old}} - (\text{Learning Rate} \times \text{Gradient})
   $$

This continuous feedback loop is repeated over many epochs until the model converges to optimal parameters.

📖 **Further Reading:** [GeeksforGeeks - Backpropagation in Neural Networks](https://www.geeksforgeeks.org/machine-learning/backpropagation-in-neural-network/)

In [22]:
# let's build a tiny trainable model ourselves before nn.Linear or nn.Module or CNN's etc.
x = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor([2.0, 4.0, 6.0])

weight = torch.tensor(0.0, requires_grad=True)

y_pred = weight * x

print(y_pred)
print(f"Our model currenly predicts: {y_pred} because w = 0")

print("\nWe will use MSE: ")
loss = ((y_pred - y_true) ** 2).mean()
print(f"Loss is: {loss:.4f} and we can clearly see the loss is too much because predictions are very large")

loss.backward()
print(weight.grad)

print("Now we can see weight grad is negative because out weight is 0 and we need to make it 2")
print("if grad>0 it will decrease and if grad <0 it will increase right now that's the case.")



tensor([0., 0., 0.], grad_fn=<MulBackward0>)
Our model currenly predicts: tensor([0., 0., 0.], grad_fn=<MulBackward0>) because w = 0

We will use MSE: 
Loss is: 18.6667 and we can clearly see the loss is too much because predictions are very large
tensor(-18.6667)
Now we can see weight grad is negative because out weight is 0 and we need to make it 2
if grad>0 it will decrease and if grad <0 it will increase right now that's the case.


### 📉 Mathematical Optimization: Gradient Descent

To update our parameters, we apply the standard parameter update formula:
$$
w \leftarrow w - \eta \frac{\partial L}{\partial w}
$$

Where:
* **$w$**: The model parameter (e.g., weight) we want to optimize.
* **$L$**: The scalar loss value measuring model performance.
* **$\eta$** (eta): The **learning rate**, controlling the step size of each modification.

---

### 🚫 The Role of `torch.no_grad()`

When we update parameters manually, we wrap our calculations in a `with torch.no_grad():` block.

**Why is this necessary?**
* **Prevents Tracking:** PyTorch automatically records every mathematical operation on tensors with `requires_grad=True` in its computational graph.
* **Avoids Graph Pollution:** Updating the weights is an administrative optimization step, not part of the model's forward path. We must perform this update without updating the graph itself.

In [23]:
learning_rate = 0.1

with torch.no_grad():
  weight = weight - learning_rate * weight.grad

print(weight)
print("Now weight shifts from 0 to correct one if we change learning rate we can reduce the loss")



tensor(1.8667)
Now weight shifts from 0 to correct one if we change learning rate we can reduce the loss


### ⚠️ Critical Warning: Gradient Accumulation

In PyTorch, gradients are **not** overwritten or reset to zero when you call `.backward()`. Instead, they are added to the existing values stored in the `.grad` attribute.

$$
\text{Stored Gradient}_{t} = \text{Stored Gradient}_{t-1} + \text{New Gradient}_{t}
$$

#### Why does PyTorch do this?
Accumulating gradients allows you to compute the loss over extremely large effective batch sizes that cannot fit inside your GPU's VRAM at once.

#### How to handle this in standard training:
If you do not manually clear old gradients, your updates will become corrupted by historical step values. To prevent this, always reset your parameters' gradients before executing your backward pass:

```python
# Reset gradients to zero
optimizer.zero_grad()

# Compute new gradients
loss.backward()
```

### 🔄 The Consolidated Model Execution Pipeline

This complete conceptual sequence represents a single iteration of training in PyTorch:

```
               ┌───────────────┐
               │  Input Data   │
               └───────┬───────┘
                       ▼
               ┌───────────────┐
               │  Model Step   │ ◄─── (Weights & Biases)
               └───────┬───────┘
                       ▼
               ┌───────────────┐
               │  Predictions  │
               └───────┬───────┘
                       ▼
               ┌───────────────┐
               │   Loss Step   │ ◄─── (Compares with Ground Truth)
               └───────┬───────┘
                       ▼
               ┌───────────────┐
               │ .backward()   │ ◄─── (Computes Gradients)
               └───────┬───────┘
                       ▼
               ┌───────────────┐
               │ Update Step   │ ◄─── (Optimizer / manual update)
               └───────────────┘
```

---

### 🛠️ PyTorch's Core Architecture

PyTorch coordinates this pipeline using five fundamental pillars:

| Pillar | Responsibility |
| :--- | :--- |
| **Tensor** | The multi-dimensional array storing model data and parameters. |
| **`requires_grad=True`** | Instructs PyTorch to track operations on this tensor for gradient computation. |
| **Autograd Engine** | The background engine recording operations to build the computational graph. |
| **`.backward()`** | The trigger that traverses the graph in reverse to calculate derivatives. |
| **`.grad`** | The attribute where the calculated derivatives are stored for parameter updates. |